# 01 - Explore Competition Data

**Goal:** Understand the benchmark dataset — question types, difficulty distribution, answer formats.

**Competition:** [NVIDIA Nemotron Model Reasoning Challenge](https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import sys

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))
from src.data_prep import load_competition_data, explore_data

RAW_DIR = Path('../data/raw')
print('Files in data/raw:', list(RAW_DIR.glob('*')) if RAW_DIR.exists() else 'EMPTY - download data first')

## Load Data

If data/raw is empty, run:
```bash
kaggle competitions download -c nvidia-nemotron-model-reasoning-challenge -p data/raw
```

In [ ]:
# List all files downloaded
if RAW_DIR.exists():
    for f in sorted(RAW_DIR.rglob('*')):
        if f.is_file():
            size_mb = f.stat().st_size / 1024 / 1024
            print(f'{f.relative_to(RAW_DIR)}  ({size_mb:.2f} MB)')
else:
    print('No data directory found')

In [ ]:
# Try loading train data
try:
    train_df = load_competition_data('train')
    print(f'Train data shape: {train_df.shape}')
    print(f'Columns: {list(train_df.columns)}')
    print(f'\nData types:')
    print(train_df.dtypes)
except FileNotFoundError as e:
    print(e)

In [ ]:
# Explore data summary
try:
    summary = explore_data(train_df)
    print(f"Total problems: {summary['num_rows']}")
    print(f"Columns: {summary['columns']}")
    print(f"\nSample problems:")
    for i, row in enumerate(summary['sample']):
        print(f"\n--- Problem {i+1} ---")
        for k, v in row.items():
            print(f"  {k}: {str(v)[:200]}")
except NameError:
    print('No data loaded yet')

## Category/Type Distribution

In [ ]:
# Distribution analysis
try:
    for col in train_df.columns:
        if train_df[col].dtype == 'object' and train_df[col].nunique() < 50:
            print(f"\n=== {col} distribution ===")
            print(train_df[col].value_counts())
except NameError:
    print('No data loaded yet')

## Answer Format Analysis

In [ ]:
# Analyze answer formats (look for answer column)
try:
    answer_cols = [c for c in train_df.columns if 'answer' in c.lower() or 'solution' in c.lower()]
    print(f'Potential answer columns: {answer_cols}')
    
    for col in answer_cols:
        answers = train_df[col].astype(str)
        print(f"\n=== {col} stats ===")
        print(f"  Unique answers: {answers.nunique()}")
        print(f"  Avg length: {answers.str.len().mean():.1f} chars")
        print(f"  Numeric answers: {answers.apply(lambda x: x.replace('.','').replace('-','').isdigit()).sum()}")
        print(f"  Sample answers: {answers.head(10).tolist()}")
except NameError:
    print('No data loaded yet')

## Question Length Analysis

In [ ]:
# Analyze question lengths
try:
    text_cols = [c for c in train_df.columns if 'question' in c.lower() or 'problem' in c.lower() or 'prompt' in c.lower()]
    if not text_cols:
        text_cols = [c for c in train_df.columns if train_df[c].dtype == 'object']
    
    for col in text_cols[:3]:
        lengths = train_df[col].astype(str).str.len()
        print(f"\n=== {col} length stats ===")
        print(f"  Mean: {lengths.mean():.0f} chars")
        print(f"  Median: {lengths.median():.0f} chars")
        print(f"  Min: {lengths.min()} | Max: {lengths.max()}")
        print(f"  Estimated tokens (÷4): {lengths.mean()/4:.0f} avg")
except NameError:
    print('No data loaded yet')

## Test Data Preview

In [ ]:
# Load and preview test data
try:
    test_df = load_competition_data('test')
    print(f'Test data shape: {test_df.shape}')
    print(f'Columns: {list(test_df.columns)}')
    test_df.head(3)
except FileNotFoundError as e:
    print(e)

## Key Findings

**TODO:** Fill in after running with data:
- Total problems: 
- Question types: 
- Answer format distribution: 
- Average question length: 
- Hardest categories (to target with LoRA): 